In [ ]:
import sys
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import f1_score

In [ ]:
from itertools import product
from vix_price.balck_scholes.bs_pricers import bs_call, bs_vega
from vix_price.balck_scholes.implied_volatility import implied_volatility, IV_METHODS




In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

## Progress bar function

In [ ]:

import time


def print_progress_bar(iteration, total, bar_length=50):
    """
    Prints a progress bar to the console.

    Parameters:
    - iteration: Current iteration (int).
    - total: Total iterations (int).
    - bar_length: Character length of the bar (int).
    """

    # Calculate progress
    progress = float(iteration) / float(total)
    
    n_full_bars = int(progress * bar_length)
    spaces = bar_length - n_full_bars

    # Build the progress bar string
    bar = '█' * n_full_bars +  ' ' * spaces
    
    # Calculate percentage completion
    percent = round(progress * 100, 1)

    # Print progress bar
    sys.stdout.write(f"\rProgress: {bar} {percent}% Complete")
    sys.stdout.flush()

#  testing
for i in range(100):
    print_progress_bar(i + 1, 100)
    time.sleep(0.1)

In [ ]:

kks = np.arange(0.1, 1.4, 0.01)
vols = np.arange(0.05, 2.5, 0.05)
tts = np.arange(1 / 48, 1, 1 / 48)
rs = np.arange(0.01, 0.1, 0.01)
# S = 1
#
# vol = 0.55

# plt.plot(kks, bs_vega(1, kks, 1/12, 0., vol));
# r = 0.
# T = 1. / 12
# S = 1

search_space = list(product(vols, kks, tts, rs))

# randomly shuffle the search space
np.random.shuffle(search_space)

results = []

N = 50 * 10 ** 3

for i, (vol, k, t, r) in enumerate(search_space[:N]):
    print_progress_bar(i + 1, N)
    call_price1 = bs_call(s=1, k=k, t=t, r=r, sigma=vol)
    call_price2 = bs_call(s=1, k=k, t=t, r=r, sigma=vol, method='norm')
    vega = bs_vega(s=1, k=k, t=t, r=r, sigma=vol)
    for method in ['iterative', 'brentq', 'newton']:
        err = ''
        try:
            iv1 = implied_volatility(call_price1, S=1, K=k, T=t, r=r, method=method)
            iv2 = implied_volatility(call_price2, S=1, K=k, T=t, r=r, method=method)
        except Exception as e:
            err = e
            iv = np.nan
        result_dict = {
            "method": method,
            "vol": vol,
            "k": k,
            "t": t,
            "r": r,
            'vega': vega,
            'call_price': call_price1,
            'call_price2': call_price2,
            "iv": iv1,
            "iv1": iv2,
            "rel. diff": abs(vol - iv1) / max(vol, iv1),
            "rel. diff2": abs(vol - iv2) / max(vol, iv2),
            "error": err
        }
        results.append(result_dict)

df = pd.DataFrame(results)

In [ ]:
df

In [ ]:
import seaborn as sns

x_cols = ['vol', 'k', 't', 'r']

# subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for i, col in enumerate(x_cols):
    sns.lineplot(data=df, x=col, y='rel. diff', hue='method', ax=axes[i // 2, i % 2])
    
    # # round x ticks up to 2 digits and rotate vertical
    # x_tciks = axes[i // 2, i % 2].get_xticks()
    # axes[i // 2, i % 2].set_xticklabels([f'{x:.2f}' for x in x_tciks], rotation=90)
    
    axes[i // 2, i % 2].set_title(f'Relative difference vs {col}')
    axes[i // 2, i % 2].set_xlabel(col)
    axes[i // 2, i % 2].set_ylabel('Relative difference')
    
plt.show()

In [ ]:
import seaborn as sns

x_cols = ['vol', 'k', 't', 'r']

# subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for i, col in enumerate(x_cols):
    sns.lineplot(data=df, x=col, y='rel. diff2', hue='method', ax=axes[i // 2, i % 2])
    
    # # round x ticks up to 2 digits and rotate vertical
    # x_tciks = axes[i // 2, i % 2].get_xticks()
    # axes[i // 2, i % 2].set_xticklabels([f'{x:.2f}' for x in x_tciks], rotation=90)
    
    axes[i // 2, i % 2].set_title(f'Relative difference vs {col}')
    axes[i // 2, i % 2].set_xlabel(col)
    axes[i // 2, i % 2].set_ylabel('Relative difference')
    
plt.show()

In [ ]:
# plot rel. diff vs rel. diff2
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

x_name = 'vol'
y_name = 'call_price'

sns.lineplot(data=df, x=x_name, y=y_name,  ax=ax)

ax.set_title(f'{x_name} vs {y_name}')
ax.set_xlabel(x_name)
ax.set_ylabel(y_name)

plt.show()

In [ ]:
# models to predict rel. diff based on `method`, `k`, `t`, `vol`

X, y = df[['method', 'k', 't', 'vol']], df['rel. diff'].fillna(1.)

# quantize the target variable

q3 = 0.1
q4 = 0.25
q5 = 0.5



y = pd.qcut(y, [0, q3, q4, q5, 1], labels=False, duplicates='drop')

# rename methods to 0 1 2
X['method'] = X['method'].map({'iterative': 0, 'brentq': 1, 'newton': 2})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:


lr = LogisticRegression(random_state=42)
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_pred_rf = rf.predict(X_test)

# print f1 scores
print(f'Linear Regression F1: {f1_score(y_test, y_pred_lr, average="weighted")}')
print(f'Random Forest F1: {f1_score(y_test, y_pred_rf, average="weighted")}')

In [ ]:
# plot predictions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))


lr_cm = pd.crosstab(y_test, y_pred_lr, rownames=['Actual'], colnames=['Predicted'])
rf_cm = pd.crosstab(y_test, y_pred_rf, rownames=['Actual'], colnames=['Predicted'])

sns.heatmap(lr_cm, annot=True, cmap='Blues', fmt='d', ax=axes[0])
sns.heatmap(rf_cm, annot=True, cmap='Blues', fmt='d', ax=axes[1])

axes[0].set_title('Linear Regression')
axes[1].set_title('Random Forest')

plt.show()


# plot ceoffs and feature importances

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(x=X.columns, y=lr.coef_[0], ax=axes[0])
sns.barplot(x=X.columns, y=rf.feature_importances_, ax=axes[1])

axes[0].set_title('Linear Regression')
axes[1].set_title('Random Forest')

plt.show()


In [ ]:
fail_df = df[df['rel. diff'].isna()]

In [ ]:
fail_df

In [ ]:
def leap_year(year):
    return (year%4==0)-(year%100==0)+(year%400==0)

In [ ]:
leap_year(2104)

In [ ]:
f = lambda x: 1. / (x ** 4 + x ** 3 + x ** 2 + x + 1)


def plot_area_aunder(low, up, func, min_x, max_x):
    min_, max_ = min(low, min_x), max(up, max_x)
    area_under = integ.quad(func, low, up)
    x = np.linspace(min_, max_, 1000)
    y = func(x)

    plt.plot(x, y, 'r')
    plt.fill_between(x, y, where=(x > low) & (x < up), alpha=0.3)
    plt.title(f"Area under the curve: {area_under[0]:.2f}")
    plt.show()


import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual

# interactive plot
interact(plot_area_aunder, low=(-10, 10, 0.1), up=(-10, 10, 0.1), func=fixed(f), min_x=fixed(-500), max_x=fixed(500))
